# Monthly gambling fundamentals — FLUT, DKNG and CZR

Read supporting and contrary evidence together. This review shows **12 explicit months** and **comparable rolling three-month windows**, keeping market demand, brand position, gross hold and casino accounting separate. No composite rating or profit forecast is produced.

The endpoint is selected in `config/current_snapshot.json`: July 2026 means August 2025–July 2026 monthly observations and May–July 2026 as the latest three months. Growth compares exactly the same months one year earlier. Changes from the prior monthly read use this **same retained capture**, not information known at that earlier date.

[Saved monthly review](../docs/monthly_fundamentals_20260912.md) · [Dated issuer and ownership context](../docs/company_context_20260912.md) · [Workflow](../docs/industry_workflow.md)


In [ ]:
as_of = None  # Current UTC; earlier cutoffs reject later operating/mapping/context captures.
end_month = None  # None uses the explicitly selected through_month; never automatically newest.
ny_weeks = 8
export_note = False
note_destination = None  # Optional absolute NEW dated .md path in an existing directory.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import pandas as pd
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import hashlib

ROOT = Path.cwd().resolve()
if (ROOT / "gaming" / "src").is_dir():
    ROOT = ROOT / "gaming"
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from variant_gaming.refresh import load_validated_snapshot, frozen_database_sha256
selection = json.loads((ROOT / "config/current_snapshot.json").read_text())
DB = ROOT / selection["database_file"]
cutoff = as_of or datetime.now(timezone.utc).isoformat()
snapshot = load_validated_snapshot(root=ROOT, database=DB, as_of=cutoff,
                                   expected_sha256=selection["database_sha256"])
observations, coverage = snapshot["observations"], snapshot["coverage"]
manifest = snapshot["manifest"]
before = snapshot["binding"]["database_sha256"]
pd.set_option("display.max_colwidth", None)
print("Capture:", manifest["finished_at"], "| Information cutoff:", cutoff)
print("Explicit snapshot:", DB)


from variant_gaming.industry import (
    build_industry_tables, operator_scope, coverage_table, CZR_MAPPING_SOURCES,
    research_note, export_research_note,
)
from variant_gaming.monthly_review import build_monthly_review, fundamentals_update
from variant_gaming.flut_scorecard import scorecard_scope
from variant_gaming.legal import load_legal_events
end_month = end_month or selection["through_month"]
endpoint = pd.Period(end_month, freq="M")
if endpoint.end_time.date() > pd.Timestamp(cutoff).date():
    raise ValueError("Requested month has not ended by the information cutoff")
quarter = str(endpoint.asfreq("Q-DEC"))


In [ ]:
# Verify mapping evidence and the explicitly dated qualitative context before use.
context_receipt = json.loads((ROOT / "docs/company_context_sources_20260912.json").read_text())
for binding in [*CZR_MAPPING_SOURCES, *context_receipt["sources"]]:
    captured = pd.Timestamp(binding["captured_at"])
    if pd.isna(captured) or captured.tzinfo is None or captured > pd.Timestamp(cutoff):
        raise ValueError("Mapping/context source was captured after the cutoff or has an invalid clock")
    source_path = (ROOT / binding["source_file"]).resolve()
    if not source_path.is_relative_to((ROOT / "data/raw").resolve()):
        raise ValueError("Mapping/context source leaves retained raw data")
    if hashlib.sha256(source_path.read_bytes()).hexdigest() != binding["source_sha256"]:
        raise ValueError("Mapping/context source hash mismatch")
print("Issuer/ownership context reviewed:", context_receipt["reviewed_on"], "— this vintage does not update when end_month changes.")


In [ ]:
# Reconcile native history once; all endpoints reuse those monthly rows.
tables = build_industry_tables(observations, quarter=quarter, through_month=end_month, ny_weeks=ny_weeks)
review = build_monthly_review(tables, end_month=end_month)
monthly, rolling3, assessment = review["monthly"], review["rolling3"], review["assessment"]
update = fundamentals_update(review)
legal_events = load_legal_events(ROOT / "config/legal_events.json", root=ROOT, as_of=cutoff)
# Fixed Q2 2026 context from the verified September 12 source review, not the trend engine.
issuer_context = {
    "FLUT": "Dated Q2 2026 issuer counterevidence: US revenue -6% YoY; US adjusted EBITDA $119m, -70%.",
    "DKNG": "Dated Q2 2026 issuer counterevidence: Sports Consumer Volume +14.5%, total revenue -4.6%; adjusted EBITDA $114.6m versus $300.6m.",
    "CZR": "Dated Q2 2026 issuer counterevidence: Digital revenue +2.3%, but Digital adjusted EBITDA -15%; land-based segments differ.",
}
for company, context in issuer_context.items():
    mask = update.subject.eq(company)
    update.loc[mask, "contrary_evidence"] += " | " + context + " Different period and scope from the state windows; see the dated company context."
print("12 monthly reads:", str(endpoint-11), "through", end_month)
print("Latest 3M:", str(endpoint-2), "through", end_month, "versus the same months one year earlier")
display(update[["subject", "window", "supporting_evidence", "contrary_evidence", "what_changed", "interpretation", "uncertainty", "next_check"]])


## Latest three months — demand and competitive position

Official statewide handle measures wagering volume, while a brand's share describes position within that state. Growth and share are computed from **sums of all three requested months**, not averages of percentages. One missing or excluded source month blocks the comparison; it does not shorten the window. Amounts are USD, growth/share are percent, and changes are percentage points.


In [ ]:
display(assessment.query("company == 'MARKET' and metric == 'handle'")[["state_code", "expected_months", "amount", "prior_amount", "yoy_growth_pct", "growth_rate_change_pp", "growth_rate_direction", "status"]].round(3))
display(assessment.query("company != 'MARKET' and metric == 'handle'")[["company", "state_code", "amount", "prior_amount", "yoy_growth_pct", "share_pct", "prior_share_pct", "share_change_pp", "status"]].round(3))


## Twelve months of evidence

Lines break where comparisons are unavailable. A valid current amount remains in the table even when its prior-year counterpart is missing. The covered states are not a national index. The rolling view reduces dependence on a single month, but does not establish a persistent causal driver.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for frame, ax, title in [(monthly, axes[0], "Official handle — monthly YoY"), (rolling3, axes[1], "Official handle — rolling 3M YoY")]:
    for state, group in frame.query("company == 'MARKET' and metric == 'handle'").groupby("state_code"):
        group = group.sort_values("end_month")
        ax.plot(pd.to_datetime(group.end_month), group.yoy_growth_pct, marker="o", label=state)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set(title=title, ylabel="YoY growth (%)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for state, ax in zip(["MA", "MI"], axes):
    for company, group in rolling3.query("company != 'MARKET' and state_code == @state and metric == 'handle'").groupby("company"):
        group = group.sort_values("end_month")
        ax.plot(pd.to_datetime(group.end_month), group.share_change_pp, marker="o", label=company)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set(title=f"{state} brand handle share — rolling 3M change", ylabel="YoY share change (pp)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## Gross hold and casino — separate economics

Sportsbook gross hold is native gross revenue divided by handle for the same window. Higher hold is an outcome, not proof of stronger demand, retention or sustainable earnings. Michigan Gross Receipts and Adjusted Gross remain separate. Casino has no sportsbook handle/hold measure. Issuer net revenue and adjusted EBITDA use different accounting and geographic scopes.


In [ ]:
display(assessment.query("vertical == 'online_sports_betting' and metric == 'gross_revenue'")[["company", "state_code", "native_metric", "amount", "yoy_growth_pct", "hold_pct", "prior_hold_pct", "hold_change_pp", "status"]].round(3))
display(assessment.query("vertical == 'online_casino'")[["company", "native_metric", "expected_months", "amount", "prior_amount", "yoy_growth_pct", "share_pct", "share_change_pp", "growth_rate_change_pp", "status"]].round(3))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for frame, ax, title in [(monthly, axes[0], "MI casino — monthly YoY"), (rolling3, axes[1], "MI casino — rolling 3M YoY")]:
    for label, group in frame.query("company == 'MARKET' and vertical == 'online_casino'").groupby("native_metric"):
        group = group.sort_values("end_month")
        ax.plot(pd.to_datetime(group.end_month), group.yoy_growth_pct, marker="o", label=label)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set(title=title, ylabel="YoY growth (%)")
    ax.legend(); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()


## Rate changes and the prior monthly read

**Acceleration/deceleration here means one specified comparison:** latest three-month YoY growth minus the preceding **non-overlapping** three-month YoY growth. At the July endpoint, that is May–July versus February–April rates, each already compared with its exact prior-year months. Every month in all four windows must qualify.

**Change from the prior monthly read** instead compares the July endpoint with June: May–July versus April–June rolling YoY rates. Those windows overlap. This is a change between two calculations on today's retained capture, not a claim about what an analyst could have known in June.


In [ ]:
display(assessment[(assessment.metric.eq("handle")) | assessment.vertical.eq("online_casino")][["company", "state_code", "native_metric", "end_month", "yoy_growth_pct", "preceding3_end_month", "preceding3_yoy_growth_pct", "growth_rate_change_pp", "previous_read_end_month", "previous_read_yoy_growth_pct", "read_change_yoy_pp", "share_change_pp", "previous_read_share_change_pp", "read_change_share_pp"]].round(3))


## Gaps, native identities and visible monthly values

MA January 2025 contains a $0.90 disagreement within its original PDF: the main online handle total differs from comparative totals and the operator sum. The existing one-cent reconciliation gate stays in place. January 2026 handle YoY and January–March 2026 rolling handle/hold comparisons therefore remain unavailable; the latest May–July comparison is unaffected.

CZR's reviewed Michigan mapping uses Grand Traverse sportsbook from May 2021 and both Grand Traverse/Sault casino licenses from July 2024. Earlier ownership-crossing comparisons stay unavailable. Exactly one native alias per license/month is required; Sault sportsbook is not attributed. Actual source labels are preserved below. FLUT excludes International; DKNG's panel excludes Golden Nugget; CZR digital evidence does not describe its land-based group.


In [ ]:
display(operator_scope())
display(review["coverage"])
display(monthly[["company", "state_code", "native_metric", "end_month", "amount", "prior_amount", "yoy_growth_pct", "share_pct", "share_change_pp", "hold_pct", "current_status", "prior_status"]].round(3))
gaps = pd.concat([monthly.assign(read="monthly"), rolling3.assign(read="rolling3")], ignore_index=True)
display(gaps[gaps.status.ne("comparable")][["company", "state_code", "native_metric", "end_month", "read", "expected_months", "prior_months", "missing_months", "prior_missing_months", "gap_details"]])


## New York — timely weekly evidence

NY's recent complete Monday–Sunday weeks remain separate. NY cash-basis GGR and weekly handle are not prorated into months or blended into the rolling panel. Sports calendars can affect adjacent weeks; missing weeks and unreconciled populations remain visible.


In [ ]:
weekly = tables["weekly"]
if weekly.empty:
    print("New York weekly evidence is unavailable.")
else:
    display(weekly.drop(columns="source_refs").round(3))


## Dated issuer, tax and legal context

The [September 12 primary-source review](../docs/company_context_20260912.md) gives Q2 2026 issuer economics, Caesars ownership/operating evidence, NJ/IL tax context and the exact MA source disagreement. Its Q2 figures are a **fixed dated vintage**; changing the operating endpoint does not make them newer. Issuer-reported causes are not causes established by the state panel. New Jersey's completed-event statewide statistics remain separate.

The small legal register below retains the source-established status date, effective date, observed fact, inference and recheck needs. Capture time does not confirm that an older legal status remains current. Proposals/interim rulings are not final nationwide outcomes.


In [ ]:
display(pd.DataFrame([{"company": company, "context_vintage": "Q2 2026 / reviewed 2026-09-12", "issuer_evidence": fact} for company, fact in issuer_context.items()]))
display(pd.DataFrame(context_receipt["sources"])[["source_id", "published_on", "captured_at", "source_url", "source_file", "source_sha256"]])
display(legal_events[["event_id", "jurisdiction", "status", "status_as_of", "effective_date", "observed_fact", "accounting_effect", "business_interpretation", "limitations", "recheck_status", "next_check"]])


## Source drilldown and broader collection coverage

All native monthly inputs and source versions remain available. The operating source list covers every requested current/prior window, including excluded evidence. The capture's 34 stored state/product series do not establish nationwide analytical coverage; series not selected for that refresh retained their earlier evidence.


In [ ]:
needed_months = {m.strip() for field in ["expected_months", "prior_months"] for value in rolling3[field] for m in value.split(",")}
window_inputs = review["inputs"][review["inputs"].period_start.str[:7].isin(needed_months)]
display(window_inputs.drop(columns="source_refs"))
refs = sorted({ref for group in [monthly, rolling3, weekly] for values in group.source_refs for ref in values})
source_table = pd.DataFrame(refs, columns=["source_url", "source_file", "source_sha256"])
display(source_table)
display(scorecard_scope(observations))
display(coverage_table(observations, coverage, capture_at=manifest["finished_at"]))
display(pd.read_csv(DB.parent / "collection_summary.csv"))


## Optional dated note

Export remains disabled unless an explicit new dated path is supplied. The note records supporting/contrary evidence, comparison windows, prior-read meaning, scope limits and source references. It records descriptive research, not a forecast or valuation approval.


In [ ]:
note = research_note(update, quarter=quarter, through_month=end_month,
    capture_at=manifest["finished_at"], database_sha256=before, sources=source_table, legal_events=legal_events)
note += "\n## Dated issuer and ownership context\n\nQ2 2026 issuer context reviewed September 12, 2026; this vintage does not roll forward with the operating endpoint. Source-qualified review: " + "[dated company context](" + (ROOT / "docs/company_context_20260912.md").as_posix() + ")" + "\n"
note += "\n" + "\n".join(f"- {row['source_id']}: [{row['source_file']}]({row['source_url']}) — `{row['source_sha256']}`; captured {row['captured_at']}." for row in context_receipt["sources"]) + "\n"
assert frozen_database_sha256(DB) == before, "Database changed during review"
if export_note:
    if note_destination is None:
        raise ValueError("Set an absolute new dated note_destination before enabling export")
    print("Saved:", export_research_note(note, note_destination))
else:
    print("Monthly research note prepared in memory; export disabled.")
